In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
from datetime import datetime, timedelta


chars = "0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ ,-"
PAD_IDX, SOS_IDX, EOS_IDX = 0, 1, 2

char2idx = {char: idx + 3 for idx, char in enumerate(chars)}
char2idx["<PAD>"] = PAD_IDX
char2idx["<SOS>"] = SOS_IDX
char2idx["<EOS>"] = EOS_IDX
idx2char = {idx: char for char, idx in char2idx.items()}
VOCAB_SIZE = len(char2idx)

def generate_data():
    start_date = datetime(1900, 1, 1)
    random_days = random.randint(0, 45000)
    random_date = start_date + timedelta(days=random_days)
    return random_date.strftime("%B %d, %Y"), random_date.strftime("%Y-%m-%d")

def string_to_tensor(s):
    # Convert string to list of IDs, and add <EOS> at the end
    ids = [char2idx[c] for c in s] + [EOS_IDX]
    return torch.tensor(ids, dtype=torch.long).unsqueeze(0) # Shape: [1, Time]

In [2]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)

    def forward(self, x):
        embedded = self.embedding(x)
        outputs, hidden = self.gru(embedded)
        return hidden # The Context Vector

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, current_char, hidden):
        current_char = current_char.unsqueeze(1) # Shape: [1, 1]
        embedded = self.embedding(current_char)
        output, new_hidden = self.gru(embedded, hidden)
        prediction = self.fc_out(output.squeeze(1)) # Shape: [1, Vocab_Size]
        return prediction, new_hidden


In [5]:
encoder = Encoder(VOCAB_SIZE)
decoder = Decoder(VOCAB_SIZE)
optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.0005)
criterion = nn.CrossEntropyLoss()

def train(iterations=2000, forcing_ratio=0.5):
    print("--- Starting Training ---")
    for i in range(1, iterations + 1):
        input_str, target_str = generate_data()
        input_tensor = string_to_tensor(input_str)
        target_tensor = string_to_tensor(target_str)

        optimizer.zero_grad()
        loss = 0

        encoder_hidden = encoder(input_tensor)

        decoder_hidden = encoder_hidden
        decoder_input = torch.tensor([SOS_IDX])

        for t in range(target_tensor.size(1)):
            prediction, decoder_hidden = decoder(decoder_input, decoder_hidden)

            loss += criterion(prediction, target_tensor[:, t])

            if random.random() < forcing_ratio:
                decoder_input = target_tensor[:, t]
            else:
                top_guess = prediction.argmax(1)
                decoder_input = top_guess

        loss.backward()
        optimizer.step()

        if i % 500 == 0:
            print(f"Iteration {i}/{iterations} | Loss: {loss.item()/target_tensor.size(1):.4f}")

def evaluate(test_string):
    print(f"\n--- Evaluating: '{test_string}' ---")
    with torch.no_grad():
        input_tensor = string_to_tensor(test_string)
        encoder_hidden = encoder(input_tensor)

        decoder_hidden = encoder_hidden
        decoder_input = torch.tensor([SOS_IDX])

        decoded_chars =[]

        for _ in range(15):
            prediction, decoder_hidden = decoder(decoder_input, decoder_hidden)
            top_guess = prediction.argmax(1).item()

            if top_guess == EOS_IDX:
                break

            decoded_chars.append(idx2char[top_guess])
            decoder_input = torch.tensor([top_guess])

        print("Prediction: ", "".join(decoded_chars))

In [6]:
train(iterations=15000, forcing_ratio=0.5)
evaluate("October 31, 1999")
evaluate("January 01, 2026")

--- Starting Training ---
Iteration 500/15000 | Loss: 1.2035
Iteration 1000/15000 | Loss: 1.2160
Iteration 1500/15000 | Loss: 0.9983
Iteration 2000/15000 | Loss: 1.0062
Iteration 2500/15000 | Loss: 0.6958
Iteration 3000/15000 | Loss: 0.7155
Iteration 3500/15000 | Loss: 0.7157
Iteration 4000/15000 | Loss: 0.4838
Iteration 4500/15000 | Loss: 0.3888
Iteration 5000/15000 | Loss: 0.3596
Iteration 5500/15000 | Loss: 0.3700
Iteration 6000/15000 | Loss: 0.3707
Iteration 6500/15000 | Loss: 0.2765
Iteration 7000/15000 | Loss: 0.2451
Iteration 7500/15000 | Loss: 0.1034
Iteration 8000/15000 | Loss: 0.2263
Iteration 8500/15000 | Loss: 0.2699
Iteration 9000/15000 | Loss: 0.1611
Iteration 9500/15000 | Loss: 0.2403
Iteration 10000/15000 | Loss: 0.2355
Iteration 10500/15000 | Loss: 0.1690
Iteration 11000/15000 | Loss: 0.2396
Iteration 11500/15000 | Loss: 0.1814
Iteration 12000/15000 | Loss: 0.1463
Iteration 12500/15000 | Loss: 0.0642
Iteration 13000/15000 | Loss: 0.1451
Iteration 13500/15000 | Loss: 0.

In [19]:
evaluate("December 29, 2025")


--- Evaluating: 'December 29, 2025' ---
Prediction:  2025-12-29
